# Optimisation
This notebook takes the same model as in the previous one
and demonstrates how an optimisation package can be used to 
obtain _one possible set_ of parameters that could be used
to represent the observed data.

In [ ]:
%pip install summerepi2==1.3.6
%pip install estival
%pip install nevergrad

import pandas as pd
import numpy as np
import nevergrad as ng

import plotly.graph_objects as go
pd.options.plotting.backend = "plotly"

from estival import targets as est
from estival import priors as esp
from estival.model import BayesianCompartmentalModel
from estival.wrappers.nevergrad import optimize_model
from summer2 import CompartmentalModel
from summer2.parameters import Parameter
from summer2.functions.time import get_linear_interpolation_function

### Preparing the model
This follows the approach to construction in the previous notebook.
Using a population of one million as the approximate population of a Victorian Public Health Network (given there are six of them)

In [ ]:
total_population = 1e6
infectious_seed = Parameter("infectious_seed")
run_period = [0.0, 15.0]
infect_comps = ["infectious_asympt", "infectious_sympt"]
model_comps = ["vaccinated", "susceptible", "recovered"] + infect_comps
sir_model = CompartmentalModel(times=run_period, compartments=model_comps, infectious_compartments=infect_comps, timestep=0.2)
mask_start_time = Parameter("mask_start_time")
mask_scale_time = Parameter("mask_scale_time")
mask_end_time = mask_start_time + mask_scale_time
coverage = get_linear_interpolation_function([mask_start_time, mask_end_time], [0.0, Parameter("face_mask_coverage")])
infection_rate = Parameter("contact_rate") * (1.0 - coverage * Parameter("face_mask_efficacy"))
sir_model.add_infection_frequency_flow(name="infection", contact_rate=infection_rate, source="susceptible", dest="infectious_asympt")
vacc_infection_rate = infection_rate * (1.0 - Parameter("vacc_efficacy"))
sir_model.add_infection_frequency_flow(name="infection_vacc", contact_rate=vacc_infection_rate, source="vaccinated", dest="infectious_asympt")
progression_rate = Parameter("recovery_rate") * 2.0
sir_model.add_transition_flow(name="progression", fractional_rate=progression_rate, source="infectious_asympt", dest="infectious_sympt")
resolve_sympt_rate = progression_rate + Parameter("isolation_rate")
sir_model.add_transition_flow(name="recovery", fractional_rate=resolve_sympt_rate, source="infectious_sympt", dest="recovered")
suscept_pop = total_population - infectious_seed
start_pop = {
    "susceptible": suscept_pop * (1.0 - Parameter("vacc_coverage")),
    "vaccinated": suscept_pop * Parameter("vacc_coverage"),
    "infectious_asympt": infectious_seed,
}
sir_model.set_initial_population(start_pop)
sir_model.request_output_for_flow("cases", "progression")
parameters = {
    "isolation_rate": 0.0,
    "vacc_efficacy": 0.0,
    "vacc_coverage": 0.0,
}

## Calibration
Set priors using simple uniform distributions and optimise with 1000 iterations using `nevergrad`.
Daily case notification numbers for a Victorian PHN provided by Bethany

In [ ]:
cases = pd.Series([5, 12, 28, 57, 120, 215, 312, 290, 230, 165, 98, 50, 20, 8])
targets = [est.TruncatedNormalTarget("cases", cases, (0.0, np.inf), 20.0)]

In [ ]:
priors = [
    esp.UniformPrior("contact_rate", (0.5, 1.5)),
    esp.UniformPrior("recovery_rate", (0.1, 0.8)),
    esp.UniformPrior("face_mask_coverage", (0.0, 1.0)),
    esp.UniformPrior("face_mask_efficacy", (0.0, 1.0)),
    esp.UniformPrior("infectious_seed", (0.0, 10.0)),
    esp.UniformPrior("mask_start_time", (3.0, 10.0)),
    esp.UniformPrior("mask_scale_time", (1.0, 7.0)),
]
bcm = BayesianCompartmentalModel(sir_model, parameters, priors, targets)
opt_class = ng.optimizers.TwoPointsDE
orunner = optimize_model(bcm, opt_class=opt_class)
rec = orunner.minimize(1000)

## Report optimised parameters

In [ ]:
opti_params = rec.value[1]
opti_params

## Inspect outputs

In [ ]:
sir_model.run(parameters | opti_params)
base_outs = sir_model.get_derived_outputs_df()
fig = base_outs.plot()
fig.add_trace(go.Scatter(x=cases.index, y=cases))

## Intervention comparison

In [ ]:
intervention_params = {"face_mask_coverage": 1.0}
sir_model.run(parameters | opti_params | intervention_params)
int_outs = sir_model.get_derived_outputs_df()["cases"]
fig.add_trace(go.Scatter(x=int_outs.index, y=int_outs))